# Dataset Preparation


## Explanation of the code snippet

### Handle categories
- categorical_features = [...]: Tạo một danh sách chứa tên của 3 cột dữ liệu phân loại (ví dụ: Giới tính, Loại gói cước, Thời hạn hợp đồng).

- Vòng lặp for & hàm .unique(): Vòng lặp này đi qua từng cột trong danh sách trên và in ra các giá trị khác nhau (duy nhất) tồn tại trong cột đó.

- Mục đích: Đây là bước kiểm tra (sanity check) để bạn nhìn tổng quan xem mỗi cột đang chứa những nhóm nào. Ví dụ, nó có thể in ra: Gender : ['Female' 'Male'] giúp bạn biết cột này chỉ có 2 loại giá trị.

### One-hot encoding (for tree-based models)
- Hàm `pd.get_dummies()` của thư viện Pandas thực hiện kỹ thuật One-Hot Encoding. Kỹ thuật này sẽ tách mỗi giá trị chữ thành một cột riêng biệt, đánh số 1 nếu có và 0 nếu không có.
- Các tham số quan trọng:

  - columns=categorical_features: Chỉ định rõ Pandas chỉ thực hiện biến đổi trên 3 cột đã liệt kê, các cột số khác (như tuổi, cước phí...) sẽ được giữ nguyên.

  - drop_first=True: Đây là tham số cực kỳ quan trọng để tránh "Bẫy biến giả" (Dummy Variable Trap) hay hiện tượng đa cộng tuyến. Tham số này sẽ tự động xóa đi cột đầu tiên được tạo ra cho mỗi biến.

    - Ví dụ minh họa: Nếu cột Gender có 2 giá trị là Male và Female. Nếu không có drop_first, hàm sẽ tạo ra 2 cột mới là Gender_Male và Gender_Female. Tuy nhiên, điều này là dư thừa. Nếu một người có Gender_Male = 1, ta chắc chắn người đó không phải là Nữ. Ngược lại, nếu Gender_Male = 0, máy tính có thể tự suy ra người đó là Nữ.

    - Do đó, drop_first=True sẽ bỏ đi cột Gender_Female. Việc này giúp giảm bớt số lượng cột trong tập dữ liệu, làm cho mô hình học nhanh hơn và bớt nhiễu hơn.

Ghi chú về dòng comment # (for tree-based models):
Việc dùng One-hot encoding với drop_first=True mang tính bắt buộc đối với các mô hình tuyến tính (như Linear Regression, Logistic Regression). Đối với các mô hình dạng cây (như Decision Tree, Random Forest, XGBoost), hiện tượng đa cộng tuyến không làm hỏng mô hình, nên đôi khi người ta không cần dùng drop_first=True, hoặc thậm chí dùng kỹ thuật khác (Label Encoding). Tuy nhiên, cách bạn đang viết vẫn hoàn toàn chuẩn xác và mô hình cây vẫn sẽ học rất tốt với dữ liệu này.

## Pre-processing and feature engineering

In [39]:
def process_customer_data(input_path):
  # Load data
  df = pd.read_csv(input_path)
  
  # handle missing values
  print(df.isnull().sum())

  # Drop rows with missing values
  df_processed = df.dropna()
  print(df_processed.isnull().sum())

  # Count how many rows are duplicates base on CustomerID
  duplicate_count = df_processed.duplicated(subset=['CustomerID']).sum()
  print(f'Number of duplicate rows based on CustomerID: {duplicate_count}')

  # View actual duplicate rows
  duplicates = df_processed[df_processed.duplicated(subset=['CustomerID'], keep=False)]
  print(duplicates.sort_values(by='CustomerID').head(10))

  # Keep the first occurrence and remove subsequent ones
  df_processed = df_processed.drop_duplicates(subset=['CustomerID'], keep='first')

  # Convert datatype
  df_processed['CustomerID'] = df_processed['CustomerID'].astype('int64')
  int_column = ['Age', 'Tenure', 'Support Calls', 'Last Interaction']
  for col in int_column:
      df_processed[col] = df_processed[col].astype(int)

  # Handle categories
  categorical_features = ['Gender', 'Subscription Type', 'Contract Length']
  for col in categorical_features:
      print(f"{col} : {df_processed[col].unique()}")

  # One-hot encoding (for tree-based models)
  df_processed = pd.get_dummies(df_processed, columns=categorical_features, drop_first=True)

  # Feature engineering
  df_processed['Tenure_Age_Ratio'] = df_processed['Tenure'] / (df_processed['Age'] + 1)
  df_processed['Spend_per_Usage'] = df_processed['Total Spend'] / (df_processed['Usage Frequency'] + 1)
  df_processed['Support_Calls_per_Tenure'] = df_processed['Support Calls'] / (df_processed['Tenure'] + 1)

  # Create customer segments based on spending
  df_processed['Spending_Group'] = pd.qcut(df_processed['Total Spend'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])

  # Create tenure groups
  df_processed['Tenure_Group'] = pd.cut(df_processed['Tenure'], 
                                        bins=[0, 12, 24, 36, 100],
                                        labels=['<1yr', '1-2yrs', '2-3yrs', '3+yrs'])

  # Add categorical features to the list
  categorical_features.extend(['Spending_Group', 'Tenure_Group'])

  return df_processed

## Loop split dataset into 10 parts

In [40]:

# Export
import os
out_dir = "../ml-ops_customer-churn-prediction/data/processed"
os.makedirs(out_dir, exist_ok=True)

for i in range(1, 11):
  input_file = f"../ml-ops_customer-churn-prediction/data/raw/train_period_{i}.csv"

  output_file = os.path.join(out_dir, f"train_period_{i}_processed.csv")
  df_result = process_customer_data(input_file)
  df_result.to_csv(output_file, index=False)
  print(f"Period {i} is done")

CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64
CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64
Number of duplicate rows based on CustomerID: 0
Empty DataFrame
Columns: [CustomerID, Age, Gender, Tenure, Usage Frequency, Support Calls, Payment Delay, Subscription Type, Contract Length, Total Spend, Last Interaction, Churn]
Index: []
Gender : <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Subscription Type : <StringArray>
['Standard', 'Basic', 'Premium']
Length: 3, dtype: str
Contract Length : <StringArray>

In [41]:
all_data = []

for i in range(1, 11):
  processed_file = os.path.join(out_dir, f"train_period_{i}_processed.csv")
  df = pd.read_csv(processed_file)
  all_data.append(df)

# Combine all into a single DataFrame
df_combined = pd.concat(all_data, ignore_index=True)

combined_output = os.path.join(out_dir, "processed.csv")
df_combined.to_csv(combined_output, index=False)

print(f"Combined 10 file into 1 file: {df_combined.shape}")

Combined 10 file into 1 file: (440832, 19)


In [42]:
df_processed = pd.read_csv("./data/processed/processed.csv")
df.head()

C:\Users\KietKlat\AppData\Local\Temp\ipykernel_28056\216236712.py:1: DtypeWarning: Columns (0: Contract Length_Monthly) have mixed types. Specify dtype option on import or set low_memory=False.
  df_processed = pd.read_csv("./data/processed/processed.csv")


,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn,Gender_Male,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Quarterly,Tenure_Age_Ratio,Spend_per_Usage,Support_Calls_per_Tenure,Spending_Group,Tenure_Group
0,403865,45,3,16.0,3,17.0,613.30,9,0.0,True,True,False,True,0.065217,36.076471,0.750000,Low,<1yr
1,403866,49,15,12.0,4,4.0,576.97,20,0.0,True,False,False,False,0.300000,44.382308,0.250000,Low,1-2yrs
2,403867,35,26,17.0,2,9.0,763.60,25,0.0,True,True,False,True,0.722222,42.422222,0.074074,High,2-3yrs
3,403868,41,51,6.0,3,11.0,696.06,11,0.0,False,False,True,False,1.214286,99.437143,0.057692,Medium,3+yrs
4,403869,44,3,15.0,1,6.0,565.33,9,0.0,False,True,False,False,0.066667,35.333125,0.250000,Low,<1yr
